<a href="https://colab.research.google.com/github/manmeet3591/insar/blob/main/Code/5.%20ML%20inSAR%20Pipeline/ML_inSAR_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# """
# ML-based InSAR Deformation Analysis for Permian Basin
# ====================================================
# This script implements an image-to-image machine learning pipeline for InSAR deformation prediction
# using OPERA Level 3 displacement data for the Permian Basin area in Texas.

# The model takes 2-channel input (displacement measurements from two time periods) and
# produces 1-channel output (predicted future displacement).
# """

In [ ]:
!pip install netCDF4

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import netCDF4 as nc
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import pandas as pd
from datetime import datetime, timedelta
import seaborn as sns
from tqdm.notebook import tqdm
import re

In [ ]:
print("TensorFlow Version:", tf.__version__)

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Set up directories using your Google Drive mount
WORKDIR = '/content/drive/MyDrive/insar_ml_project'
DATADIR = f'{WORKDIR}/data'
PROCESSEDDIR = f'{WORKDIR}/processed'
MODELDIR = f'{WORKDIR}/models'
RESULTDIR = f'{WORKDIR}/results'

# Create directories if they don't exist
os.makedirs(DATADIR, exist_ok=True)
os.makedirs(PROCESSEDDIR, exist_ok=True)
os.makedirs(MODELDIR, exist_ok=True)
os.makedirs(RESULTDIR, exist_ok=True)

print(f"Directories set up in Google Drive:")
print(f"- Data: {DATADIR}")
print(f"- Processed: {PROCESSEDDIR}")
print(f"- Models: {MODELDIR}")
print(f"- Results: {RESULTDIR}")

In [ ]:
def load_single_opera_nc_file(file_path):
    """
    Load displacement data from a single OPERA NetCDF file.

    Parameters:
    -----------
    file_path : str
        Path to the NetCDF file

    Returns:
    --------
    data_dict : dict or None
        Dictionary containing displacement data and metadata, or None if error.
    """
    print(f"Loading NetCDF file: {file_path}")
    if not os.path.exists(file_path):
        print(f"Error: File not found at {file_path}")
        return None

    try:
        # Open NetCDF file
        dataset = nc.Dataset(file_path, 'r')

        # Extract metadata
        metadata = {}
        for attr in dataset.ncattrs():
            metadata[attr] = getattr(dataset, attr)
        print("\n--- File Metadata ---")
        for k, v in metadata.items():
            print(f"- {k}: {v}")

        # Extract acquisition time
        acquisition_time = None
        if 'time' in dataset.variables:
            time_var = dataset.variables['time']
            if hasattr(time_var, 'units'):
                time_units = time_var.units
                if 'since' in time_units:
                    base_date_str = time_units.split('since ')[1].strip()
                    # Try parsing with date and time, fall back to date only
                    try:
                        base_date = datetime.strptime(base_date_str, '%Y-%m-%d %H:%M:%S')
                    except ValueError:
                        try:
                            base_date = datetime.strptime(base_date_str, '%Y-%m-%d %H:%M:%S.%f')
                        except ValueError:
                            base_date = datetime.strptime(base_date_str, '%Y-%m-%d')
                    time_values = time_var[:]
                    if 'days' in time_units:
                        acquisition_time = [base_date + timedelta(days=float(t)) for t in time_values]
                    elif 'seconds' in time_units:
                        acquisition_time = [base_date + timedelta(seconds=float(t)) for t in time_values]
                    else:
                        acquisition_time = time_values
                else:
                    acquisition_time = time_var[:]
            else:
                acquisition_time = time_var[:]
        else:
            filename = os.path.basename(file_path)
            match = re.search(r'(\d{8}T\d{6})', filename)
            if match:
                time_str = match.group(1)
                acquisition_time = datetime.strptime(time_str, '%Y%m%dT%H%M%S')

        metadata['acquisition_time'] = acquisition_time
        print(f"- Acquisition Time: {acquisition_time}")

        # Extract displacement data
        displacement = None
        disp_var_name = None
        potential_disp_vars = ['displacement', 'disp', 'los_displacement', 'velocity']
        for var_name in potential_disp_vars:
            if var_name in dataset.variables:
                displacement = dataset.variables[var_name][:]
                disp_var_name = var_name
                break

        if displacement is None:
            # Look for variable with 'displacement' in name if specific names not found
            disp_vars = [var for var in dataset.variables if 'disp' in var.lower()]
            if disp_vars:
                displacement = dataset.variables[disp_vars[0]][:]
                disp_var_name = disp_vars[0]
            else:
                raise ValueError(f"No displacement data found in {file_path}")

        print(f"- Displacement variable found: {disp_var_name}")
        print(f"- Displacement data shape: {displacement.shape}")

        # Handle different array shapes (e.g., time dimension)
        if len(displacement.shape) == 3:
            print("Displacement data has 3 dimensions (likely time, lat, lon). Selecting first time slice.")
            displacement = displacement[0, :, :]
            print(f"- Shape after selecting first slice: {displacement.shape}")
        elif len(displacement.shape) != 2:
            raise ValueError(f"Displacement data has unexpected shape: {displacement.shape}. Expected 2D (lat, lon) or 3D (time, lat, lon).")

        # Extract quality data if available
        quality = None
        quality_vars = ['quality', 'coherence', 'uncertainty']
        for var_name in quality_vars:
            if var_name in dataset.variables:
                quality = dataset.variables[var_name][:]
                print(f"- Quality variable found: {var_name}, Shape: {quality.shape}")
                break

        # Extract geolocation data
        geolocation = {}
        geo_vars = ['latitude', 'longitude', 'lat', 'lon', 'x', 'y']
        for var_name in geo_vars:
            if var_name in dataset.variables:
                geolocation[var_name] = dataset.variables[var_name][:]
                print(f"- Geolocation variable found: {var_name}, Shape: {geolocation[var_name].shape}")

        # Close NetCDF file
        dataset.close()

    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None

    # Create data dictionary
    data_dict = {
        'displacement': displacement,
        'quality': quality,
        'geolocation': geolocation,
        'metadata': metadata
    }

    print("\nData loaded successfully.")
    return data_dict

# Load your specific .nc file
nc_file_path = '/content/drive/MyDrive/insar_ml_project/data/nc_data.nc'

# Load the data
data_dict = load_single_opera_nc_file(nc_file_path)

# Check if data loaded successfully
if data_dict:
    displacement_map = data_dict['displacement']
    print(f"\nDisplacement map shape: {displacement_map.shape}")
else:
    print("\nFailed to load data. Please check the file path and format.")

In [ ]:
# Preprocess and Visualize Raw Data

# This cell visualizes the raw displacement map loaded from the NetCDF file.

In [ ]:
def visualize_raw_displacement(displacement_map, title="Raw Displacement Map", save_path=None):
    """
    Visualize the raw displacement map.
    """
    if displacement_map is None:
        print("No displacement map to visualize.")
        return

    plt.figure(figsize=(10, 8))
    plt.imshow(displacement_map, cmap='RdBu_r')
    plt.colorbar(label='Displacement')
    plt.title(title)
    plt.xlabel("Longitude Index")
    plt.ylabel("Latitude Index")
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Raw displacement map saved to: {save_path}")
    plt.show()

# Visualize the loaded displacement map
if data_dict:
    visualize_raw_displacement(displacement_map, save_path=f"{RESULTDIR}/raw_displacement_map.png")

In [ ]:
# Create ML Dataset from Single File
# Since we only have one displacement map, we need to generate multiple training samples.
# This cell implements a strategy using overlapping spatial patches.

# - Input (X): Two overlapping patches (2 channels).
# - Output (y): The second patch (1 channel).

In [ ]:
def create_patched_dataset(displacement_map, patch_size=64, stride=32):
    """
    Create input-output pairs using overlapping patches from a single displacement map.

    Parameters:
    -----------
    displacement_map : numpy.ndarray
        2D displacement map (height, width)
    patch_size : int
        Size of the square patches
    stride : int
        Stride for sliding the patches

    Returns:
    --------
    X : numpy.ndarray
        Input data with shape (n_samples, patch_size, patch_size, 2)
    y : numpy.ndarray
        Target data with shape (n_samples, patch_size, patch_size, 1)
    """
    if displacement_map is None:
        print("Displacement map is not loaded. Cannot create dataset.")
        return None, None

    height, width = displacement_map.shape
    X_list = []
    y_list = []

    print(f"Creating patches of size {patch_size}x{patch_size} with stride {stride}...")

    # Iterate through the map with the specified stride
    for r in tqdm(range(0, height - patch_size - stride + 1, stride), desc="Rows"):
        for c in range(0, width - patch_size - stride + 1, stride):
            # Extract two overlapping patches
            patch1 = displacement_map[r:r+patch_size, c:c+patch_size]
            patch2 = displacement_map[r+stride:r+stride+patch_size, c+stride:c+stride+patch_size]

            # Ensure patches have the correct size (handle edge cases if necessary, though stride logic should prevent this)
            if patch1.shape == (patch_size, patch_size) and patch2.shape == (patch_size, patch_size):
                # Input: Stack the two patches as channels
                X_sample = np.stack([patch1, patch2], axis=-1)

                # Output: The second patch
                y_sample = patch2[..., np.newaxis] # Add channel dimension

                X_list.append(X_sample)
                y_list.append(y_sample)

    if not X_list:
        print("Warning: No patches were created. Check map dimensions, patch size, and stride.")
        return None, None

    # Convert to numpy arrays
    X = np.array(X_list)
    y = np.array(y_list)

    print(f"Dataset created with {X.shape[0]} samples.")
    print(f"Input shape (X): {X.shape}")  # (n_samples, patch_size, patch_size, 2)
    print(f"Target shape (y): {y.shape}") # (n_samples, patch_size, patch_size, 1)

    return X, y



PATCH_SIZE = 64 # Adjust as needed, should be divisible by 2 multiple times for U-Net
STRIDE = 32     # Adjust overlap as needed

if data_dict:
    X, y = create_patched_dataset(displacement_map, patch_size=PATCH_SIZE, stride=STRIDE)
else:
    X, y = None, None

In [ ]:
# Normalize Data
# Normalize the input (X) and target (y) patches to the [0, 1] range.

In [ ]:
def normalize_data(X, y):
    """
    Normalize input and output data to [0, 1] range.
    """
    if X is None or y is None:
        print("Data not available for normalization.")
        return None, None, None

    print("Normalizing data...")
    # Compute min and max values for each channel in X
    X_min = np.min(X, axis=(0, 1, 2), keepdims=True)
    X_max = np.max(X, axis=(0, 1, 2), keepdims=True)

    # Compute min and max values for y
    y_min = np.min(y)
    y_max = np.max(y)

    # Avoid division by zero if max == min
    X_range = X_max - X_min
    y_range = y_max - y_min
    X_range[X_range == 0] = 1e-10  # Add small epsilon
    if y_range == 0:
        y_range = 1e-10

    # Normalize X and y to [0, 1] range
    X_norm = (X - X_min) / X_range
    y_norm = (y - y_min) / y_range

    # Store normalization parameters
    norm_params = {
        'X_min': X_min,
        'X_max': X_max,
        'y_min': y_min,
        'y_max': y_max
    }

    print(f"Normalization parameters:")
    print(f"- X_min: {X_min.flatten()}")
    print(f"- X_max: {X_max.flatten()}")
    print(f"- y_min: {y_min}")
    print(f"- y_max: {y_max}")

    return X_norm, y_norm, norm_params


X_norm, y_norm, norm_params = normalize_data(X, y)

# Display a sample normalized patch
if X_norm is not None:
    print("\nSample Normalized Input Patch (Channel 1):")
    plt.figure(figsize=(5, 5))
    plt.imshow(X_norm[0, :, :, 0], cmap='gray')
    plt.colorbar()
    plt.title("Normalized Input Patch (Ch 1)")
    plt.show()

    print("\nSample Normalized Target Patch:")
    plt.figure(figsize=(5, 5))
    plt.imshow(y_norm[0, :, :, 0], cmap='gray')
    plt.colorbar()
    plt.title("Normalized Target Patch")
    plt.show()

In [ ]:
# Split Data
# Split the generated dataset into training, validation, and test sets.

In [ ]:
def split_data(X_norm, y_norm, test_size=0.2, val_size=0.2):
    """
    Split data into train, validation, and test sets.
    """
    if X_norm is None or y_norm is None:
        print("Normalized data not available for splitting.")
        return None, None, None, None, None, None

    print("Splitting data...")
    # First split into train+val and test
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X_norm, y_norm, test_size=test_size, random_state=42
    )

    # Then split train+val into train and val
    # Adjust val_size relative to the size of train_val set
    relative_val_size = val_size / (1.0 - test_size)
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=relative_val_size, random_state=42
    )

    print(f"Total samples: {X_norm.shape[0]}")
    print(f"Train set: {X_train.shape[0]} samples ({X_train.shape[0]/X_norm.shape[0]*100:.1f}%)")
    print(f"Validation set: {X_val.shape[0]} samples ({X_val.shape[0]/X_norm.shape[0]*100:.1f}%)")
    print(f"Test set: {X_test.shape[0]} samples ({X_test.shape[0]/X_norm.shape[0]*100:.1f}%)")

    return X_train, y_train, X_val, y_val, X_test, y_test


X_train, y_train, X_val, y_val, X_test, y_test = split_data(X_norm, y_norm)

In [ ]:
# Define U-Net Model
# Define the U-Net architecture for the image-to-image task.

In [ ]:
def build_unet_model(input_shape, filters_base=64):
    """
    Build U-Net model for image-to-image translation.
    """
    # Input layer
    inputs = layers.Input(input_shape)

    # Encoder (downsampling path)
    # Level 1
    conv1 = layers.Conv2D(filters_base, 3, activation='relu', padding='same')(inputs)
    conv1 = layers.Conv2D(filters_base, 3, activation='relu', padding='same')(conv1)
    pool1 = layers.MaxPooling2D(pool_size=(2, 2))(conv1)

    # Level 2
    conv2 = layers.Conv2D(filters_base*2, 3, activation='relu', padding='same')(pool1)
    conv2 = layers.Conv2D(filters_base*2, 3, activation='relu', padding='same')(conv2)
    pool2 = layers.MaxPooling2D(pool_size=(2, 2))(conv2)

    # Level 3
    conv3 = layers.Conv2D(filters_base*4, 3, activation='relu', padding='same')(pool2)
    conv3 = layers.Conv2D(filters_base*4, 3, activation='relu', padding='same')(conv3)
    pool3 = layers.MaxPooling2D(pool_size=(2, 2))(conv3)

    # Level 4
    conv4 = layers.Conv2D(filters_base*8, 3, activation='relu', padding='same')(pool3)
    conv4 = layers.Conv2D(filters_base*8, 3, activation='relu', padding='same')(conv4)
    pool4 = layers.MaxPooling2D(pool_size=(2, 2))(conv4)

    # Bottom level
    conv5 = layers.Conv2D(filters_base*16, 3, activation='relu', padding='same')(pool4)
    conv5 = layers.Conv2D(filters_base*16, 3, activation='relu', padding='same')(conv5)

    # Decoder (upsampling path)
    # Level 4
    up6 = layers.Conv2DTranspose(filters_base*8, 2, strides=(2, 2), padding='same')(conv5)
    concat6 = layers.Concatenate()([up6, conv4])
    conv6 = layers.Conv2D(filters_base*8, 3, activation='relu', padding='same')(concat6)
    conv6 = layers.Conv2D(filters_base*8, 3, activation='relu', padding='same')(conv6)

    # Level 3
    up7 = layers.Conv2DTranspose(filters_base*4, 2, strides=(2, 2), padding='same')(conv6)
    concat7 = layers.Concatenate()([up7, conv3])
    conv7 = layers.Conv2D(filters_base*4, 3, activation='relu', padding='same')(concat7)
    conv7 = layers.Conv2D(filters_base*4, 3, activation='relu', padding='same')(conv7)

    # Level 2
    up8 = layers.Conv2DTranspose(filters_base*2, 2, strides=(2, 2), padding='same')(conv7)
    concat8 = layers.Concatenate()([up8, conv2])
    conv8 = layers.Conv2D(filters_base*2, 3, activation='relu', padding='same')(concat8)
    conv8 = layers.Conv2D(filters_base*2, 3, activation='relu', padding='same')(conv8)

    # Level 1
    up9 = layers.Conv2DTranspose(filters_base, 2, strides=(2, 2), padding='same')(conv8)
    concat9 = layers.Concatenate()([up9, conv1])
    conv9 = layers.Conv2D(filters_base, 3, activation='relu', padding='same')(concat9)
    conv9 = layers.Conv2D(filters_base, 3, activation='relu', padding='same')(conv9)

    # Output layer (linear activation for regression)
    outputs = layers.Conv2D(1, 1, activation='linear')(conv9)

    # Create model
    model = models.Model(inputs=inputs, outputs=outputs)

    return model

# Build and compile the model
if X_train is not None:
    input_shape = X_train.shape[1:]  # (patch_size, patch_size, 2)
    model = build_unet_model(input_shape)

    # Compile model (using Adam optimizer and MSE loss for regression)
    model.compile(
        optimizer=optimizers.Adam(learning_rate=1e-4),
        loss=mean_squared_error,
        metrics=[MeanAbsoluteError()]
    )

    print("U-Net Model Summary:")
    model.summary()
else:
    print("Training data not available. Cannot build model.")
    model = None

In [ ]:
# Train Model
# Train the U-Net model using the prepared datasets.

In [ ]:
def train_model_notebook(model, X_train, y_train, X_val, y_val, batch_size=16, epochs=50):
    """
    Train the U-Net model (notebook version).
    """
    if model is None or X_train is None:
        print("Model or training data not available. Cannot train.")
        return None, None

    print("Starting model training...")
    # Define callbacks
    model_path = f"{MODELDIR}/best_unet_model.h5"

    checkpoint = ModelCheckpoint(
        model_path,
        monitor='val_loss',
        save_best_only=True,
        mode='min',
        verbose=1
    )

    early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=15,  # Reduced patience for potentially smaller dataset
        restore_best_weights=True,
        verbose=1
    )

    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,  # Reduced patience
        min_lr=1e-6,
        verbose=1
    )

    # Train model
    history = model.fit(
        X_train, y_train,
        batch_size=batch_size,
        epochs=epochs,
        validation_data=(X_val, y_val),
        callbacks=[checkpoint, early_stopping, reduce_lr],
        verbose=1
    )

    print(f"Training complete. Best model saved to {model_path}")
    return model, history

# Train the model
BATCH_SIZE = 16  # Adjust based on Colab resources
EPOCHS = 50      # Adjust as needed, early stopping will prevent overfitting

if model is not None:
    trained_model, history = train_model_notebook(model, X_train, y_train, X_val, y_val, batch_size=BATCH_SIZE, epochs=EPOCHS)
else:
    trained_model, history = None, None

In [ ]:
# Plot Training History
# Visualize the training and validation loss/MAE over epochs.

In [ ]:
def plot_training_history(history, save_dir=RESULTDIR):
    """
    Plot training history.
    """
    if history is None:
        print("No training history available to plot.")
        return

    print("Plotting training history...")
    # Create figure
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # Plot loss
    ax = axes[0]
    ax.plot(history.history['loss'], label='Training Loss')
    ax.plot(history.history['val_loss'], label='Validation Loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss (MSE)')
    ax.set_title('Training and Validation Loss')
    ax.legend()
    ax.grid(True)

    # Plot MAE
    ax = axes[1]
    ax.plot(history.history['mean_absolute_error'], label='Training MAE')
    ax.plot(history.history['val_mean_absolute_error'], label='Validation MAE')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Mean Absolute Error')
    ax.set_title('Training and Validation MAE')
    ax.legend()
    ax.grid(True)

    # Save figure
    save_path = f"{save_dir}/training_history.png"
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Training history plot saved to: {save_path}")

# Plot the history
plot_training_history(history)

In [ ]:
# Evaluate Model
# Evaluate the trained model on the test set and calculate performance metrics.

In [ ]:
def denormalize_data(y_norm, norm_params):
    """
    Denormalize data using stored normalization parameters.
    """
    if y_norm is None or norm_params is None:
        return None
    y = y_norm * (norm_params['y_max'] - norm_params['y_min'] + 1e-10) + norm_params['y_min']
    return y


def evaluate_model_notebook(model, X_test, y_test, norm_params):
    """
    Evaluate model performance on test data (notebook version).
    """
    if model is None or X_test is None:
        print("Model or test data not available for evaluation.")
        return None, None

    print("Evaluating model on test set...")

    # Load the best saved model weights
    best_model_path = f"{MODELDIR}/best_unet_model.h5"

    if os.path.exists(best_model_path):
        print(f"Loading best model from: {best_model_path}")
        model.load_weights(best_model_path)
    else:
        print("Warning: Best model file not found. Evaluating with current model weights.")

    # Make predictions
    y_pred_norm = model.predict(X_test)

    # Denormalize predictions and ground truth
    y_pred = denormalize_data(y_pred_norm, norm_params)
    y_true = denormalize_data(y_test, norm_params)

    if y_pred is None or y_true is None:
        print("Error during denormalization. Cannot calculate metrics.")
        return None, None

    # Calculate metrics
    mse = mean_squared_error(y_true.flatten(), y_pred.flatten())
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true.flatten(), y_pred.flatten())
    r2 = r2_score(y_true.flatten(), y_pred.flatten())

    # Store metrics
    metrics = {
        'mse': mse,
        'rmse': rmse,
        'mae': mae,
        'r2': r2
    }

    print("\nModel Performance on Test Set:")
    print(f"- MSE: {metrics['mse']:.6f}")
    print(f"- RMSE: {metrics['rmse']:.6f}")
    print(f"- MAE: {metrics['mae']:.6f}")
    print(f"- R²: {metrics['r2']:.6f}")

    # Save metrics
    metrics_df = pd.DataFrame([metrics])
    metrics_save_path = f"{RESULTDIR}/test_metrics.csv"
    metrics_df.to_csv(metrics_save_path, index=False)
    print(f"\nTest metrics saved to: {metrics_save_path}")

    return metrics, y_pred_norm  # Return normalized predictions for visualization


if trained_model is not None:
    test_metrics, y_pred_norm_test = evaluate_model_notebook(trained_model, X_test, y_test, norm_params)
else:
    test_metrics, y_pred_norm_test = None, None


In [ ]:
# Visualize Results
# Visualize model predictions against ground truth for a few samples from the test set.

In [ ]:
def visualize_results_notebook(X_test, y_test, y_pred_norm, norm_params, num_samples=5, save_dir=RESULTDIR):
    """
    Visualize model predictions and ground truth (notebook version).
    """
    if X_test is None or y_test is None or y_pred_norm is None or norm_params is None:
        print("Data not available for visualization.")
        return

    print(f"Visualizing results for {num_samples} test samples...")

    # Denormalize data
    y_test_denorm = denormalize_data(y_test, norm_params)
    y_pred_denorm = denormalize_data(y_pred_norm, norm_params)

    if y_test_denorm is None or y_pred_denorm is None:
        print("Error during denormalization. Cannot visualize.")
        return

    # Select random samples
    num_samples = min(num_samples, X_test.shape[0])
    sample_indices = np.random.choice(X_test.shape[0], num_samples, replace=False)

    for i, idx in enumerate(sample_indices):
        print(f"\n--- Sample {i+1} (Index {idx}) ---")
        fig, axes = plt.subplots(1, 4, figsize=(22, 5))

        # Plot input channels (denormalized for better interpretation)
        X_test_denorm_sample = denormalize_data(X_test[idx:idx+1], norm_params)  # Denormalize single sample

        ax = axes[0]
        im = ax.imshow(X_test_denorm_sample[0, :, :, 0], cmap='RdBu_r')
        ax.set_title(f"Input Patch 1 (Index {idx})")
        plt.colorbar(im, ax=ax, label='Displacement')

        ax = axes[1]
        im = ax.imshow(X_test_denorm_sample[0, :, :, 1], cmap='RdBu_r')
        ax.set_title(f"Input Patch 2 (Index {idx})")
        plt.colorbar(im, ax=ax, label='Displacement')

        # Plot ground truth
        ax = axes[2]
        im = ax.imshow(y_test_denorm[idx, :, :, 0], cmap='RdBu_r')
        ax.set_title("Ground Truth")
        plt.colorbar(im, ax=ax, label='Displacement')

        # Plot prediction
        ax = axes[3]
        im = ax.imshow(y_pred_denorm[idx, :, :, 0], cmap='RdBu_r')
        ax.set_title("Prediction")
        plt.colorbar(im, ax=ax, label='Displacement')

        # Save figure
        save_path = f"{save_dir}/test_sample_{i+1}_prediction.png"
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"Prediction visualization saved to: {save_path}")

        # Plot error map
        error = y_pred_denorm[idx, :, :, 0] - y_test_denorm[idx, :, :, 0]
        plt.figure(figsize=(7, 6))
        plt.imshow(error, cmap='RdBu_r')
        plt.colorbar(label='Error (Prediction - Truth)')
        plt.title(f"Prediction Error (Sample {i+1})")
        error_save_path = f"{save_dir}/test_sample_{i+1}_error.png"
        plt.tight_layout()
        plt.savefig(error_save_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"Error map saved to: {error_save_path}")


visualize_results_notebook(X_test, y_test, y_pred_norm_test, norm_params, num_samples=5)


In [ ]:
# Publication Figures
# Generate combined figures suitable for publication.

In [ ]:
def create_publication_figures_notebook(X_test, y_test, y_pred_norm, norm_params, metrics, save_dir=RESULTDIR):
    """
    Create publication-quality figures (notebook version).
    """
    if X_test is None or metrics is None:
        print("Data or metrics not available for publication figures.")
        return

    print("Creating publication figures...")

    # Denormalize data
    y_test_denorm = denormalize_data(y_test, norm_params)
    y_pred_denorm = denormalize_data(y_pred_norm, norm_params)

    if y_test_denorm is None or y_pred_denorm is None:
        print("Error during denormalization. Cannot create publication figures.")
        return

    # Create combined figure for publication
    fig = plt.figure(figsize=(12, 10))
    gs = fig.add_gridspec(2, 2)

    # Plot 1 & 2: Sample prediction
    idx = np.random.randint(0, len(X_test))
    ax1 = fig.add_subplot(gs[0, 0])
    im1 = ax1.imshow(y_test_denorm[idx, :, :, 0], cmap='RdBu_r')
    ax1.set_title("(a) Ground Truth Patch")
    fig.colorbar(im1, ax=ax1, label='Displacement')

    ax2 = fig.add_subplot(gs[0, 1])
    im2 = ax2.imshow(y_pred_denorm[idx, :, :, 0], cmap='RdBu_r')
    ax2.set_title("(b) Predicted Patch")
    fig.colorbar(im2, ax=ax2, label='Displacement')

    # Plot 3: Error histogram
    ax3 = fig.add_subplot(gs[1, 0])
    error = (y_pred_denorm - y_test_denorm).flatten()
    ax3.hist(error, bins=50, alpha=0.75, density=True)
    ax3.axvline(x=0, color='r', linestyle='--')
    ax3.set_xlabel("Prediction Error (Units)")
    ax3.set_ylabel("Density")
    ax3.set_title("(c) Error Distribution")
    ax3.grid(True, linestyle=':')

    # Plot 4: Metrics table
    ax4 = fig.add_subplot(gs[1, 1])
    ax4.axis('off')
    metrics_text = (
        f"Model Performance Metrics:\n\n"
        f"RMSE: {metrics['rmse']:.3f} (Units)\n"
        f"MAE: {metrics['mae']:.3f} (Units)\n"
        f"R²: {metrics['r2']:.3f}"
    )
    ax4.text(
        0.05, 0.5, metrics_text, fontsize=12, va='center',
        bbox=dict(boxstyle='round', pad=0.5, fc='aliceblue', alpha=0.7)
    )
    ax4.set_title("(d) Performance Summary", y=0.9)

    # Save figure
    save_path_png = f"{save_dir}/publication_summary_figure.png"
    save_path_pdf = f"{save_dir}/publication_summary_figure.pdf"

    plt.tight_layout(pad=2.0)
    plt.savefig(save_path_png, dpi=300, bbox_inches='tight')
    plt.savefig(save_path_pdf, bbox_inches='tight')
    plt.show()
    print(f"Publication summary figure saved to: {save_path_png} and {save_path_pdf}")


create_publication_figures_notebook(X_test, y_test, y_pred_norm_test, norm_params, test_metrics)